In [7]:
import json
import re
from yarn_utils import YARNGraph
from itertools import permutations

# Load Data

In [29]:
FOLDER_PATH = "annotations/FRACAS_12032026/"
FILE = "5h.yarn.json"

In [30]:
with open(FOLDER_PATH + FILE) as f:
    yarn_graph_json = json.load(f)

yarn_graph = YARNGraph(yarn_graph_json)
yarn_grew = yarn_graph.grew()

In [31]:
# with open('output.json', 'w') as f:
#     json.dump(output, f)

In [32]:
yarn_grew

{'nodes': {'vi1': {'concept': 'Italy', 'type': 'V', 'var': 'vi1'},
  'vc1': {'concept': 'country', 'type': 'V', 'var': 'vc1'},
  'vr1': {'concept': 'really', 'type': 'V', 'var': 'vr1'},
  'va1': {'pred': 'ambitious-01', 'type': 'V', 'var': 'va1'},
  'vt1': {'concept': 'tenor', 'type': 'V', 'var': 'vt1'},
  's1': {'event': 's1', 'var': 's1', 'type': 'S'},
  's1-temp': {'type': 'F', 'feat': 'temp', 'var': 's1-temp'},
  's1-quant': {'type': 'F', 'feat': 'quant', 'var': 's1-quant'},
  's1-num': {'type': 'F', 'feat': 'num', 'var': 's1-num'},
  's1-def': {'type': 'F', 'feat': 'def', 'var': 's1-def'},
  'e2': {'rel': 'name', 'type': 'E', 'var': 'e2'},
  'e4': {'rel': 'ARG0', 'type': 'E', 'var': 'e4'},
  'e5': {'rel': 'mod', 'type': 'E', 'var': 'e5'},
  'e1': {'rel': 'mod', 'type': 'E', 'var': 'e1'},
  'l1': {'value': 'plural', 'var': 'l1', 'type': 'L', 'feat': 'num'},
  'l2': {'value': 'exists', 'var': 'l2', 'type': 'L', 'feat': 'quant'},
  'l3': {'value': 'indefinite', 'var': 'l3', 'type': '

# Preprocessing

In [33]:
from grewpy import Graph, GRS

grs_path = "grs/main.grs"
grs = GRS(grs_path)
yarn_grew = grs.apply(Graph(yarn_grew), strat='main')

# Build F and R

In [138]:
variables = set()
def fresh_variable(base=None):
    if base is None:
        base = 'e'
    i = 0

    while True:
        variable = base if i == 0 else f"{base}{i}"
        if variable not in variables:
            variables.add(variable)
            break
        i += 1
    return variable

In [139]:
# Create R by getting all relations between V nodes

relations = set()
for node, feats in yarn_grew['nodes'].items():
    if feats['type'] == "E":
        edge_label = feats['rel']

        for edge in yarn_grew['edges']:
            if edge['tar'] == node:
                src = edge['src']
                
            if edge['src'] == node:
                tar = edge['tar']
                

        relations.add((edge_label, src, tar))

R = {}
for relation in relations:
    label = relation[0]
    src = relation[1]
    tar = relation[2]

    if src not in R:
        R[src] = [(label, src, tar)]
    else:
        R[src].append((label, src, tar))

In [140]:
id2var = {}
F = []

for node, feats in yarn_grew['nodes'].items():
    if feats['type'] == "L" or feats['type'] == 'H':
        for edge in yarn_grew['edges']:
            if edge['src'] == node:
                tar = edge['tar']

                if yarn_grew['nodes'][tar]['type'] == 'V':
                    edge_label = feats['value'] if feats['value'] else feats['feat']
                    if 'pred' in yarn_grew['nodes'][tar]:
                        tar_label = yarn_grew['nodes'][tar]['pred']
                    else:
                        tar_label = yarn_grew['nodes'][tar]['concept']
        
                    variable = fresh_variable(base=tar_label[0])

                    F.append({
                        'id':tar,
                        'incoming_edge':node,
                        'type':"Q_"+edge_label,
                        'variable': variable,
                        'tar_label':tar_label,
                    })

                    id2var[tar] = variable
    
    if feats['type'] == 'H':
        for edge in yarn_grew['edges']:
            if edge['src'] == node:
                tar = edge['tar']

                if yarn_grew['nodes'][tar]['type'] == 'L' or yarn_grew['nodes'][tar]['type'] == 'H':
                    edge_label = feats['value'] if feats['value'] else feats['feat']

                    F.append({
                        'id':tar,
                        'incoming_edge':node,
                        'type':"Q_"+edge_label,
                        'variable': None,
                        'tar_label':None,
                    })

# Ignore definitness and number for now
F = [f for f in F if f['type'] not in ['Q_definite', 'Q_indefinite', 'Q_singular', 'Q_plural']]


In [141]:
F

[{'id': 'va2',
  'incoming_edge': 'l1',
  'type': 'Q_past',
  'variable': 'a',
  'tar_label': 'attend-01'},
 {'id': 'va1',
  'incoming_edge': 'l2',
  'type': 'Q_exists',
  'variable': 'a1',
  'tar_label': 'accountant'},
 {'id': 'vm1',
  'incoming_edge': 'l3',
  'type': 'Q_exists',
  'variable': 'm',
  'tar_label': 'meeting'},
 {'id': 'l2',
  'incoming_edge': 'h1',
  'type': 'Q_neg',
  'variable': None,
  'tar_label': None}]

In [142]:
R

{'va2': [('ARG1', 'va2', 'vm1'), ('ARG0', 'va2', 'va1')]}

# Create the Forest

In [143]:
forest = {'nodes':{}, 'edges':[]}

for i, f in enumerate(F):
    forest['nodes'][i] = {
            'id':f['id'],
            'incoming_edge':f['incoming_edge'],
            'type': f['type'],
            'variable': f['variable'],
            'tar_label': f['tar_label'],
            'relations': [f"{rel[0]}({f['variable']},{id2var[rel[2]]})" for rel in R[f['id']]] if f['id'] in R else []} # integrate R



for k1, v1 in forest['nodes'].items(): # encode specified scope
    for k2, v2 in forest['nodes'].items():
        if v1['id'] == v2['incoming_edge']:
            forest['edges'].append({'src':k1, 'rel':'', 'tar':k2})

In [144]:
forest

{'nodes': {0: {'id': 'va2',
   'incoming_edge': 'l1',
   'type': 'Q_past',
   'variable': 'a',
   'tar_label': 'attend-01',
   'relations': ['ARG1(a,m2)', 'ARG0(a,a3)']},
  1: {'id': 'va1',
   'incoming_edge': 'l2',
   'type': 'Q_exists',
   'variable': 'a1',
   'tar_label': 'accountant',
   'relations': []},
  2: {'id': 'vm1',
   'incoming_edge': 'l3',
   'type': 'Q_exists',
   'variable': 'm',
   'tar_label': 'meeting',
   'relations': []},
  3: {'id': 'l2',
   'incoming_edge': 'h1',
   'type': 'Q_neg',
   'variable': None,
   'tar_label': None,
   'relations': []}},
 'edges': [{'src': 3, 'rel': '', 'tar': 1}]}

## Add the Participant before Event constraint

In [145]:
# C (predicates are introduced after their arguments)
forest_c = forest.copy()
for src, rels in R.items():
    for rel in rels:
        tar = rel[2]

        for k1, v1 in forest_c['nodes'].items():
            for k2, v2 in forest_c['nodes'].items():
                if v1['id'] == src and v2['id'] == tar:
                    forest_c['edges'].append({'src':k2, 'rel':None, 'tar':k1})

In [146]:
forest_c

{'nodes': {0: {'id': 'va2',
   'incoming_edge': 'l1',
   'type': 'Q_past',
   'variable': 'a',
   'tar_label': 'attend-01',
   'relations': ['ARG1(a,m2)', 'ARG0(a,a3)']},
  1: {'id': 'va1',
   'incoming_edge': 'l2',
   'type': 'Q_exists',
   'variable': 'a1',
   'tar_label': 'accountant',
   'relations': []},
  2: {'id': 'vm1',
   'incoming_edge': 'l3',
   'type': 'Q_exists',
   'variable': 'm',
   'tar_label': 'meeting',
   'relations': []},
  3: {'id': 'l2',
   'incoming_edge': 'h1',
   'type': 'Q_neg',
   'variable': None,
   'tar_label': None,
   'relations': []}},
 'edges': [{'src': 3, 'rel': '', 'tar': 1},
  {'src': 2, 'rel': None, 'tar': 0},
  {'src': 1, 'rel': None, 'tar': 0}]}

# Get All Possible Trees

In [147]:
# def get_all_possible_trees(nodes):
#     chains = []

#     for perm in permutations(nodes):
#         chain = [{'src': perm[i], 'tar': perm[i+1]} for i in range(len(perm) - 1)]
#         chains.append({'edges': chain})

#     return chains

In [148]:
# nodes = list(forest_c['nodes'].keys())
# all_possible_trees = get_all_possible_trees(nodes)
# all_possible_trees

In [149]:
import itertools
import networkx as nx

def get_all_possible_trees(n):
    nodes = list(range(n))
    for seq in itertools.product(nodes, repeat=n-2):
        yield nx.from_prufer_sequence(seq)

In [150]:
# nodes = len(forest_c['nodes'].keys())
# all_possible_trees = []
# for i, tree in enumerate(get_all_possible_trees(nodes)):
#     edges = list(tree.edges())
#     print(edges)
#     edges_ascending = [{'src':edge[0], 'tar':edge[1]} for edge in edges]
#     edges_descending = [{'src':edge[1], 'tar':edge[0]} for edge in edges]
#     all_possible_trees.append({'edges': edges_ascending})
#     all_possible_trees.append({'edges': edges_descending})

In [151]:
import itertools

nodes = len(forest_c['nodes'].keys())
all_possible_trees = []

for tree in get_all_possible_trees(nodes):
    edges = list(tree.edges())

    for flips in itertools.product([False, True], repeat=len(edges)):
        directed_edges = []

        for (edge, flip) in zip(edges, flips):
            if flip:
                directed_edges.append({'src': edge[1], 'tar': edge[0]})
            else:
                directed_edges.append({'src': edge[0], 'tar': edge[1]})

        all_possible_trees.append({'edges': directed_edges})

In [152]:
all_possible_trees

[{'edges': [{'src': 0, 'tar': 1}, {'src': 0, 'tar': 2}, {'src': 0, 'tar': 3}]},
 {'edges': [{'src': 0, 'tar': 1}, {'src': 0, 'tar': 2}, {'src': 3, 'tar': 0}]},
 {'edges': [{'src': 0, 'tar': 1}, {'src': 2, 'tar': 0}, {'src': 0, 'tar': 3}]},
 {'edges': [{'src': 0, 'tar': 1}, {'src': 2, 'tar': 0}, {'src': 3, 'tar': 0}]},
 {'edges': [{'src': 1, 'tar': 0}, {'src': 0, 'tar': 2}, {'src': 0, 'tar': 3}]},
 {'edges': [{'src': 1, 'tar': 0}, {'src': 0, 'tar': 2}, {'src': 3, 'tar': 0}]},
 {'edges': [{'src': 1, 'tar': 0}, {'src': 2, 'tar': 0}, {'src': 0, 'tar': 3}]},
 {'edges': [{'src': 1, 'tar': 0}, {'src': 2, 'tar': 0}, {'src': 3, 'tar': 0}]},
 {'edges': [{'src': 0, 'tar': 2}, {'src': 0, 'tar': 1}, {'src': 1, 'tar': 3}]},
 {'edges': [{'src': 0, 'tar': 2}, {'src': 0, 'tar': 1}, {'src': 3, 'tar': 1}]},
 {'edges': [{'src': 0, 'tar': 2}, {'src': 1, 'tar': 0}, {'src': 1, 'tar': 3}]},
 {'edges': [{'src': 0, 'tar': 2}, {'src': 1, 'tar': 0}, {'src': 3, 'tar': 1}]},
 {'edges': [{'src': 2, 'tar': 0}, {'src'

In [153]:
len(all_possible_trees)

128

# Build T_all

In [154]:
# Gets the children of nodes that don't introduce variables
def get_H_children(graph):
    H_children_dict = {}
    for edge in graph['edges']:
        src = edge['src']
        tar = edge['tar']
        if not graph['nodes'][src]['variable']:
            H_children_dict[src] = H_children_dict.get(src, []) + [tar]

    return H_children_dict

In [155]:
# Get all children
def get_children(graph):
    children_dict = {}
    for edge in graph['edges']:
        src = edge['src']
        tar = edge['tar']
        children_dict[src] = children_dict.get(src, []) + [tar]
    return children_dict

# Extend children to descendants
def get_descendants(node, children_dict):
    descendants = []
    for child in children_dict.get(node, []):
        descendants.append(child)
        descendants.extend(get_descendants(child, children_dict))

    return descendants

def get_all_descendants(graph):
    children_dict = get_children(graph)
    descendants_dict = {}
    for node in children_dict:
        descendants_dict[node] = get_descendants(node, children_dict)

    return descendants_dict

In [156]:
# Constraint Checkers

def check_compatibility_of_scopes(tree, forest):
    descendants_tree = get_all_descendants(tree)
    descendants_forest = get_all_descendants(forest)

    for k,v in descendants_forest.items():
        for descendant in v:
            if k in descendants_tree:
                if descendant not in descendants_tree[k]:
                    return False
            else:
                return False
    return True

def check_locality_of_features(tree, forest):
    children_tree = get_children(tree)
    H_children_forest = get_H_children(forest)
    
    for k,v in H_children_forest.items():
        for child in v:
            if k in children_tree:
                if child not in children_tree[k]:
                    return False
            else:
                return False
    return True

In [157]:
descendants_forest = get_all_descendants(forest)
H_children_forest = get_H_children(forest)
print(descendants_forest)
print(H_children_forest)

{3: [1, 0], 2: [0], 1: [0]}
{3: [1]}


In [158]:
valid_trees = [tree for tree in all_possible_trees if check_locality_of_features(tree, forest_c)]
valid_trees = [tree for tree in valid_trees if check_compatibility_of_scopes(tree, forest_c)]
valid_trees

[{'edges': [{'src': 2, 'tar': 0}, {'src': 1, 'tar': 0}, {'src': 3, 'tar': 1}]},
 {'edges': [{'src': 1, 'tar': 0}, {'src': 2, 'tar': 1}, {'src': 3, 'tar': 1}]},
 {'edges': [{'src': 1, 'tar': 0}, {'src': 3, 'tar': 1}, {'src': 2, 'tar': 3}]},
 {'edges': [{'src': 2, 'tar': 0}, {'src': 1, 'tar': 2}, {'src': 3, 'tar': 1}]}]

In [159]:
T_all = []
for tree in valid_trees:
    new_tree = forest_c.copy()
    new_tree['edges'] = tree['edges']
    T_all.append(new_tree)

## Reformat T_all

In [160]:
# Reformat to linear tree for easier interpretation

def graph_to_linear_tree(graph):
    nodes = graph["nodes"]
    edges = graph["edges"]

    next_node = {}
    for e in edges:
        next_node[e["src"]] = e["tar"]

    all_nodes = set(nodes.keys())
    all_targets = {e["tar"] for e in edges}
    root = (all_nodes - all_targets).pop()

    def build(node_id):
        node_data = dict(nodes[node_id])

        if node_id in next_node:
            node_data["child"] = build(next_node[node_id])
        else:
            node_data["child"] = None

        return node_data

    return build(root)

In [161]:
T_all = [graph_to_linear_tree(tree) for tree in T_all]

In [162]:
T_all[0]

{'id': 'vm1',
 'incoming_edge': 'l3',
 'type': 'Q_exists',
 'variable': 'm',
 'tar_label': 'meeting',
 'relations': [],
 'child': {'id': 'va2',
  'incoming_edge': 'l1',
  'type': 'Q_past',
  'variable': 'a',
  'tar_label': 'attend-01',
  'relations': ['ARG1(a,m2)', 'ARG0(a,a3)'],
  'child': None}}

# Interpretation

In [169]:
def interpret(root, temp_variable, colors = False):
     
    if root is None:
         return ""
     
    if root['variable']:
     
        if root["type"] == "Q_exists":
            return "∃" + root["variable"] + ". (" + root["tar_label"] + "(" + root['variable'] + ") ∧ " + \
        " ∧ ".join(root['relations']) + "(" + interpret(root['child'], temp_variable) + ")"

        if root["type"] == "Q_forall":
            return "∀" + root["variable"] + ". (" + root["tar_label"] + "(" + root['variable'] + ") → " + \
        " ∧ ".join(root['relations']) + "(" + interpret(root['child'], temp_variable) + ")"

        if root["type"] == "Q_present":
            return "∃" + root["variable"] + ". (" + root["tar_label"] + "(" + root['variable'] + ") ∧ " + \
        root['variable'] + "O" + temp_variable + " ∧ " + \
        " ∧ ".join(root['relations']) + "(" + interpret(root['child'], root['variable']) + ")"

        if root["type"] == "Q_past":
            return "∃" + root["variable"] + ". (" + root["tar_label"] + "(" + root['variable'] + ") ∧ " + \
        root['variable'] + "≺" + temp_variable + " ∧ " + \
        " ∧ ".join(root['relations']) + "(" + interpret(root['child'], root['variable']) + ")"

        if root["type"] == "Q_future":
            return "∃" + root["variable"] + ". (" + root["tar_label"] + "(" + root['variable'] + ") ∧ " + \
        temp_variable + "≺" + root['variable'] + " ∧ " + \
        " ∧ ".join(root['relations']) + "(" + interpret(root['child'], root['variable']) + ")"

        if root["type"] == "Q_possibility":
            return "◇" + " (" + "∃" + root["variable"] + ". (" + root["tar_label"] + "(" + root['variable'] + ") ∧ " + \
        " ∧ ".join(root['relations']) + interpret(root["child"], temp_variable) + ")" + ")"

        if root["type"] == "Q_necessity":
            return "□" + " (" + "∃" + root["variable"] + ". (" + root["tar_label"] + "(" + root['variable'] + ") ∧ " + \
        " ∧ ".join(root['relations']) + interpret(root["child"], temp_variable) + ")" + ")"

        if root["type"] == "Q_neg":
            return "¬" + " (" + "∃" + root["variable"] + ". (" + root["tar_label"] + "(" + root['variable'] + ") ∧ " + \
        " ∧ ".join(root['relations']) + interpret(root["child"], temp_variable) + ")" + ")"
    
    else:

        if root["type"] == "Q_neg":
            return "¬" + " (" + interpret(root["child"], temp_variable) + ")"
        
        if root["type"] == "Q_possibility":
            return "◇" + " (" + interpret(root["child"], temp_variable) + ")"
        
        if root["type"] == "Q_necessity":
            return "□" + " (" + interpret(root["child"], temp_variable) + ")"

        if root["type"] == "Q_present":
            return "∃" + fresh_variable() + ". (" + root['variable'] + "O" + temp_variable + " ∧ " + \
        "(" + interpret(root['child'], root['variable']) + ")"

        if root["type"] == "Q_past":
            return "∃" + fresh_variable() + ". (" + root['variable'] + "≺" + temp_variable + " ∧ " + \
        "(" + interpret(root['child'], root['variable']) + ")"

        if root["type"] == "Q_future":
            return "∃" + fresh_variable() + ". (" + temp_variable + "≺" + root['variable'] + " ∧ " + \
        "(" + interpret(root['child'], root['variable']) + ")"
     
def clean_formula(formula):
    return formula.replace("()", "")

In [170]:
for tree in T_all:
    print(clean_formula(interpret(tree, "now")))

∃m. (meeting(m) ∧ (∃a. (attend-01(a) ∧ a≺now ∧ ARG1(a,m2) ∧ ARG0(a,a3))
∃m. (meeting(m) ∧ (∃a1. (accountant(a1) ∧ (∃a. (attend-01(a) ∧ a≺now ∧ ARG1(a,m2) ∧ ARG0(a,a3)))
∃m. (meeting(m) ∧ (¬ (∃a1. (accountant(a1) ∧ (∃a. (attend-01(a) ∧ a≺now ∧ ARG1(a,m2) ∧ ARG0(a,a3))))
¬ (∃a1. (accountant(a1) ∧ (∃m. (meeting(m) ∧ (∃a. (attend-01(a) ∧ a≺now ∧ ARG1(a,m2) ∧ ARG0(a,a3))))


In [168]:
variables

{'a', 'a1', 'a2', 'a3', 'm', 'm1', 'm2'}